# Music Studio — saved runtime and model Dataset

**Kaggle settings: Accelerator None · Internet ON.** This is an automatic,
one-cell Dataset builder. It follows the compact model-pack pattern used by
Stem Splitter, with the extra pinned Linux runtime the Music Studio launcher
actually requires. Authentication and disk checks run before large downloads;
the verified files are submitted directly to one **private Kaggle Dataset**.
Nothing passes through your PC.

1. Enable Kaggle Internet. Add private notebook Secret `KAGGLE_API_TOKEN`, or
   `KAGGLE_USERNAME` and `KAGGLE_KEY`. Never paste credentials into the cell.
2. Leave `DATASET_OWNER = ''` to use the authenticated username, or enter only
   your Kaggle username if automatic detection is unavailable.
3. Run the single code cell and wait. The runtime compilation is CPU-based and
   may take a long time. Success prints **PRIVATE MUSIC STUDIO DATASET SUBMITTED**.
4. Wait for Kaggle to process it. In **Music-Studio.ipynb**, choose **GPU T4 ×2**,
   turn on Internet, then **Add Input → Your Datasets** and attach this Dataset.

## What the Studio actually needs

The catalog has **18 exact, checksum-pinned model files** totalling
**24.77 GB**. These are model assets (not training-base weights):

- **Generation:** `generation-YuE2-3B-Q8_0.gguf`, `generation-YuE2-Vae-F32.gguf`, `companion-nar_lora_joint_v9.safetensors`
- **Covers:** `cover-SheetSage2-Q8_0.gguf`
- **Stem separation:** `stems-htdemucs_6s_fp16weights.onnx`
- **Karaoke and transcription:** `karaoke-encoder-model.int8.onnx`, `karaoke-decoder_joint-model.int8.onnx`, `karaoke-nemo128.onnx`, `karaoke-vocab.txt`, `karaoke-config.json`, `whisper-ggml-large-v3-turbo.bin`
- **Local lyric writer:** `writer-gemma-4-E4B_q4_0-it.gguf`, `writer-README.md`
- **Audio-to-MIDI:** `midi-config.json`, `midi-model.safetensors`
- **MOSS audio description:** `caption-moss-aud-f16.gguf`, `caption-moss-lm-q8_0.gguf`
- **Tempo analysis:** `tempo-beat_this.onnx`

These cover YuE2 generation and its companion/VAE, SheetSage cover generation,
HT-Demucs stems, Parakeet plus Whisper transcription, the Gemma lyric writer,
MuScriptor audio-to-MIDI, MOSS audio description, and tempo analysis. The
launcher verifies and links these catalog paths. The selected components cannot
be omitted from this single-Dataset Studio build. Artwork/video models are not
included; they are separate workflows.

## Important: CPU builder does not mean CPU runtime build

`Accelerator None` means Kaggle does not allocate a GPU to this builder. It
**still compiles** the pinned YuE2/CUDA runtime on CPU, including CUDA 13 device
code for T4/SM75. It is not a models-only download cell, and it does not make
song generation run on CPU. The normal Studio session uses the saved runtime on
GPU 0 and does not compile it again. The second T4's VRAM is not pooled.

The builder measures available disk, validates remote asset sizes before build,
checks exact bytes and SHA-256, and streams one file at a time through Kaggle's
native private Dataset-creation uploader. The largest model is
`caption-moss-lm-q8_0.gguf` (**8.71 GB**); upload
space is needed for one missing model plus reserve, not another full model set.
Only this run's own successful temporary downloads are deleted; attached
inputs and existing user files are preserved. A verified runtime from an
attached Music Studio Dataset is reused when available.

This makes one private create request; it never updates or versions an existing
Dataset. If the cell is interrupted during final submission, check Kaggle
before retrying because the outcome may be uncertain. A success message/receipt
confirms submission, not completed Kaggle processing. Build and upload logs are
under `/kaggle/working/music_studio_dataset` and the selected scratch folder.

Embedded sources SHA-256: `1b8e1c717c657e8d3f85d968bfeff99456b4896243ae7b7eda0cdc6038fa9457`
· [Setup and validation](https://github.com/bothashaun-maker/Music/blob/main/docs/music-studio/README.md)
· [Complete Studio guide](https://github.com/bothashaun-maker/Music/blob/main/Music-Studio-Guide.pdf)


In [ ]:
DATASET_OWNER = ''  # Optional: your Kaggle username, never an API key.
BUILD_JOBS = 2      # CPU compilation; lower to 1 if the build reports insufficient memory.

from pathlib import Path
import base64, hashlib, importlib, io, json, sys, zipfile

SUPPORT = Path('/kaggle/working/music-studio-dataset-support') / '1b8e1c717c65'
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '1b8e1c717c657e8d3f85d968bfeff99456b4896243ae7b7eda0cdc6038fa9457', 'Embedded Music Studio source checksum failed'
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for name in archive.namelist():
        relative = Path(name)
        if relative.is_absolute() or '..' in relative.parts or relative.parts[0] != 'music_studio':
            raise RuntimeError('Invalid embedded source path: ' + name)
        target = SUPPORT / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(name))
sys.path.insert(0, str(SUPPORT))
importlib.invalidate_caches()
for name in [n for n in sys.modules if n == 'music_studio' or n.startswith('music_studio.')]:
    sys.modules.pop(name, None)  # Rerunning this cell must not reuse stale sources.

from music_studio.common import GIB, choose_scratch
from music_studio.dataset import authenticated_api, create_dataset, username

catalog = json.loads((SUPPORT / 'music_studio/catalog.json').read_text())
api = authenticated_api()  # Check credentials before any compile, download or request.
owner = DATASET_OWNER or username(api)
if not owner:
    raise RuntimeError('Set DATASET_OWNER to your Kaggle username, then rerun. Nothing was built or downloaded.')
cache = choose_scratch(max(a['bytes'] for a in catalog['assets']) + GIB, ('/kaggle/temp', '/tmp', '/kaggle/working'), 'music-studio-downloads')
result = create_dataset(catalog, '/kaggle/working/music_studio_dataset', Path('/kaggle/input'),
                        cache, owner, jobs=BUILD_JOBS, api=api)
print('Dataset handle:', result['handle'])
print('Local receipt:', result['receipt'])
